In [1]:
import os
import time
import pandas as pd
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

# -------------------------------
# SETUP
# -------------------------------

BASE_URL = "https://nreganarep.nic.in/"
BASE_FOLDER = "MGNREGA_DATA"

os.makedirs(BASE_FOLDER, exist_ok=True)

driver = webdriver.Chrome()
wait = WebDriverWait(driver, 30)

print("Starting scraper...")

# -------------------------------
# FUNCTION: Solve Captcha + Reach Dropdown
# -------------------------------

def go_to_dropdown_page():
    driver.get(BASE_URL)
    time.sleep(2)

    # Solve captcha
    captcha_value = driver.find_element(
        By.ID, "ContentPlaceHolder1_hfCaptcha"
    ).get_attribute("value")

    driver.find_element(
        By.ID, "ContentPlaceHolder1_txtCaptcha"
    ).send_keys(captcha_value)

    driver.find_element(
        By.ID, "ContentPlaceHolder1_btnLogin"
    ).click()

    time.sleep(3)

# -------------------------------
# FUNCTION: Extract Table Data
# -------------------------------

def extract_progress_report(year, state_name):

    soup = BeautifulSoup(driver.page_source, "html.parser")
    tables = soup.find_all("table")

    if not tables:
        print("   No tables found.")
        return None

    # Choose largest table (district table)
    largest_table = max(tables, key=lambda t: len(t.find_all("tr")))

    rows = largest_table.find_all("tr")
    data = []

    for row in rows:
        cols = row.find_all("td")
        cols = [col.get_text(strip=True) for col in cols]
        if len(cols) > 5:
            data.append(cols)

    df = pd.DataFrame(data)

    if df.shape[1] < 22:
        print("   Unexpected table format.")
        return None

    # Select required columns by position
    selected_df = df.iloc[:, [1, 2, 12, 16, 17, 18, 21]].copy()

    selected_df.columns = [
        "District",
        "Household_Registered",
        "Household_Employment_Demanded",
        "Household_Employment_Availed",
        "Persons_Employment_Provided",
        "Total_Persondays",
        "Households_Completed_100_Days"
    ]

    # Remove TOTAL row
    selected_df = selected_df[
        selected_df["District"].str.upper() != "TOTAL"
    ]

    # Drop 2nd & 3rd row
    selected_df = selected_df.drop(index=[1, 2], errors="ignore")

    selected_df["Year"] = year
    selected_df["State"] = state_name

    selected_df.reset_index(drop=True, inplace=True)

    return selected_df

# -------------------------------
# GET AVAILABLE YEARS
# -------------------------------

go_to_dropdown_page()

year_dropdown = Select(
    driver.find_element(By.ID, "ContentPlaceHolder1_ddlfinyr")
)

years = [
    option.text.strip()
    for option in year_dropdown.options
    if option.text.strip() not in ["Select", ""]
]

# Optional: process from oldest to newest
years = sorted(years)
years = [year for year in years if year >= "2020-21"]

print("Years found:", years)

# -------------------------------
# MAIN LOOP
# -------------------------------

for year in years:

    print(f"\nProcessing Year: {year}")

    # Create year folder
    year_folder = os.path.join(BASE_FOLDER, year)
    os.makedirs(year_folder, exist_ok=True)

    # Restart page for this year
    go_to_dropdown_page()

    # Select year
    Select(driver.find_element(
        By.ID, "ContentPlaceHolder1_ddlfinyr"
    )).select_by_visible_text(year)

    time.sleep(2)

    # Get states
    state_dropdown = Select(
        driver.find_element(By.ID, "ContentPlaceHolder1_ddl_States")
    )

    states = [
        option.text.strip()
        for option in state_dropdown.options
        if option.text.strip() not in ["Select", "---Select---", "ALL", ""]
    ]

    for state_name in states:

        print(f"   State: {state_name}")

        try:
            # Restart page fresh for each state
            go_to_dropdown_page()

            Select(driver.find_element(
                By.ID, "ContentPlaceHolder1_ddlfinyr"
            )).select_by_visible_text(year)

            time.sleep(2)

            Select(driver.find_element(
                By.ID, "ContentPlaceHolder1_ddl_States"
            )).select_by_visible_text(state_name)

            time.sleep(3)

            # Click Progress Report
            progress_link = wait.until(
                EC.presence_of_element_located(
                    (By.XPATH, "//a[contains(., 'Progress Report')]")
                )
            )

            driver.execute_script("arguments[0].click();", progress_link)

            time.sleep(3)

            # Extract table
            df = extract_progress_report(year, state_name)

            if df is None:
                print("   Skipped due to extraction issue.")
                continue

            # Save CSV
            file_path = os.path.join(year_folder, f"{state_name}.csv")
            df.to_csv(file_path, index=False)

            print(f"   Saved: {file_path}")

            time.sleep(1)

        except Exception as e:
            print(f"   Error processing {state_name}: {e}")
            continue

print("\nScraping completed successfully.")
driver.quit()

c:\Users\Tanishq op\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\Tanishq op\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Starting scraper...
Years found: ['2021-2022', '2022-2023', '2023-2024', '2024-2025', '2025-2026']

Processing Year: 2021-2022
   State: ANDAMAN AND NICOBAR
   Saved: MGNREGA_DATA\2021-2022\ANDAMAN AND NICOBAR.csv
   State: ANDHRA PRADESH
   Saved: MGNREGA_DATA\2021-2022\ANDHRA PRADESH.csv
   State: ARUNACHAL PRADESH
   Saved: MGNREGA_DATA\2021-2022\ARUNACHAL PRADESH.csv
   State: ASSAM
   Saved: MGNREGA_DATA\2021-2022\ASSAM.csv
   State: BIHAR
   Saved: MGNREGA_DATA\2021-2022\BIHAR.csv
   State: CHHATTISGARH
   Saved: MGNREGA_DATA\2021-2022\CHHATTISGARH.csv
   State: DN HAVELI AND DD
   Saved: MGNREGA_DATA\2021-2022\DN HAVELI AND DD.csv
   State: GOA
   Saved: MGNREGA_DATA\2021-2022\GOA.csv
   State: GUJARAT
   Saved: MGNREGA_DATA\2021-2022\GUJARAT.csv
   State: HARYANA
   Saved: MGNREGA_DATA\2021-2022\HARYANA.csv
   State: HIMACHAL PRADESH
   Saved: MGNREGA_DATA\2021-2022\HIMACHAL PRADESH.csv
   State: JAMMU AND KASHMIR
   Saved: MGNREGA_DATA\2021-2022\JAMMU AND KASHMIR.csv
   State:

In [3]:
import os

BASE_FOLDER = "MGNREGA_DATA"

# Expected years (same rule you used)
years = sorted(os.listdir(BASE_FOLDER))

# Expected states (full list)
states = [
"ANDAMAN AND NICOBAR","ANDHRA PRADESH","ARUNACHAL PRADESH","ASSAM",
"BIHAR","CHHATTISGARH","GOA","GUJARAT","HARYANA","HIMACHAL PRADESH",
"JAMMU AND KASHMIR","JHARKHAND","KARNATAKA","KERALA","LADAKH",
"MADHYA PRADESH","MAHARASHTRA","MANIPUR","MEGHALAYA","MIZORAM",
"NAGALAND","ODISHA","PUNJAB","RAJASTHAN","SIKKIM","TAMIL NADU",
"TELANGANA","TRIPURA","UTTAR PRADESH","UTTARAKHAND","WEST BENGAL"
]

missing_jobs = []

for year in years:

    year_path = os.path.join(BASE_FOLDER, year)

    for state in states:

        file_path = os.path.join(year_path, f"{state}.csv")

        if not os.path.exists(file_path):
            missing_jobs.append((year, state))


print("Total missing:", len(missing_jobs))

for job in missing_jobs:
    print(job)

# Save missing list
with open("missing_jobs.txt", "w") as f:
    for year, state in missing_jobs:
        f.write(f"{year},{state}\n")

Total missing: 0
